In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import torchvision
import torchvision.transforms as transforms

In [3]:
### The data from CIFAR10 are already downloaded in the following folder
rootdir = '/opt/img/effdl-cifar10/'

# Download the training and test sets
train_dataset = torchvision.datasets.CIFAR10(root=rootdir, train=True, download=True)
test_dataset = torchvision.datasets.CIFAR10(root=rootdir, train=False, download=True)

# Extract data (images) and targets (labels)
X_train_full = train_dataset.data
y_train_full = np.array(train_dataset.targets)

X_test_full = test_dataset.data
y_test_full = np.array(test_dataset.targets)


Files already downloaded and verified
Files already downloaded and verified


In [4]:
# Flatten the data
X_train_flat = X_train_full.reshape(X_train_full.shape[0], -1)
X_test_flat = X_test_full.reshape(X_test_full.shape[0], -1)

In [5]:
from torchvision.datasets import CIFAR10
import numpy as np
import torchvision.transforms as transforms
import torch
from torch.utils.data.dataloader import DataLoader

## Normalization adapted for CIFAR10
normalize_scratch = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))

# Transforms is a list of transformations applied on the 'raw' dataset before the data is fed to the network.
# Here, Data augmentation (RandomCrop and Horizontal Flip) are applied to each batch, differently at each epoch, on the training set data only
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize_scratch,
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    normalize_scratch,
])

### Normally the data from CIFAR10 are already downloaded in the following folder from the previous lab
rootdir = '/opt/img/effdl-cifar10/'

c10train = CIFAR10(rootdir,train=True,download=True,transform=transform_train)
c10test = CIFAR10(rootdir,train=False,download=True,transform=transform_test)

trainloader = DataLoader(c10train,batch_size=32,shuffle=True)
testloader = DataLoader(c10test,batch_size=32)

Files already downloaded and verified
Files already downloaded and verified


In [6]:
## number of target samples for the final dataset
num_train_examples = len(c10train)
num_samples_subset = 15000

## We set a seed manually so as to reproduce the results easily
seed  = 2147483647

## Generate a list of shuffled indices ; with the fixed seed, the permutation will always be the same, for reproducibility
indices = list(range(num_train_examples))
np.random.RandomState(seed=seed).shuffle(indices)## modifies the list in place

## We define the Subset using the generated indices
c10train_subset = torch.utils.data.Subset(c10train,indices[:num_samples_subset])
print(f"Initial CIFAR10 dataset has {len(c10train)} samples")
print(f"Subset of CIFAR10 dataset has {len(c10train_subset)} samples")

# Finally we can define anoter dataloader for the training data
trainloader_subset = DataLoader(c10train_subset,batch_size=32,shuffle=True)
### You can now use either trainloader (full CIFAR10) or trainloader_subset (subset of CIFAR10) to train your networks.

Initial CIFAR10 dataset has 50000 samples
Subset of CIFAR10 dataset has 15000 samples


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device used:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
rootdir = "/opt/img/effdl-cifar10/"

batch_size = 32
num_samples_subset = 15000
seed = 2147483647

num_epochs = 20
learning_rate = 0.1

save_dir = "/users/local"
model_path = os.path.join(save_dir, "resnet_cifar10_subset.pth")

In [ ]:
normalize_scratch = transforms.Normalize(
    (0.4914, 0.4822, 0.4465),
    (0.2023, 0.1994, 0.2010)
)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize_scratch,
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    normalize_scratch,
])

In [ ]:
c10train = CIFAR10(
    root=rootdir,
    train=True,
    download=True,
    transform=transform_train
)

c10test = CIFAR10(
    root=rootdir,
    train=False,
    download=True,
    transform=transform_test
)

print("Training set size:", len(c10train))
print("Test set size:", len(c10test))
print("Classes:", c10train.classes)

In [ ]:
num_train_examples = len(c10train)

indices = list(range(num_train_examples))
np.random.RandomState(seed=seed).shuffle(indices)

c10train_subset = torch.utils.data.Subset(
    c10train,
    indices[:num_samples_subset]
)

print(f"Initial CIFAR10 dataset has {len(c10train)} samples")
print(f"Subset of CIFAR10 dataset has {len(c10train_subset)} samples")

In [ ]:
trainloader_subset = DataLoader(
    c10train_subset,
    batch_size=batch_size,
    shuffle=True
)

testloader = DataLoader(
    c10test,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_planes, planes,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)

        self.conv2 = nn.Conv2d(
            planes, planes,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()

        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes, planes,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = torch.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()

        self.in_planes = 64

        self.conv1 = nn.Conv2d(
            3, 64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        layers = []

        layers.append(block(self.in_planes, planes, stride))
        self.in_planes = planes

        for _ in range(1, num_blocks):
            layers.append(block(self.in_planes, planes, stride=1))

        return nn.Sequential(*layers)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.linear(out)

        return out


def ResNet18():
    return ResNet(BasicBlock, [2, 2, 2, 2])

In [ ]:
model = ResNet18().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=learning_rate,
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[10, 15],
    gamma=0.1
)

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc, all_labels, all_predictions

In [ ]:
train_losses = []
test_losses = []

train_accuracies = []
test_accuracies = []

best_test_acc = 0.0

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        trainloader_subset,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, _, _ = evaluate(
        model,
        testloader,
        criterion,
        device
    )

    scheduler.step()

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc:.4f} | "
        f"Test loss: {test_loss:.4f} | "
        f"Test acc: {test_acc:.4f}"
    )

    if test_acc > best_test_acc:
        best_test_acc = test_acc

        state = {
            "net": model.state_dict(),
            "architecture": "ResNet18",
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "num_samples_subset": num_samples_subset,
            "num_epochs": num_epochs,
            "best_test_acc": best_test_acc
        }

        torch.save(state, model_path)
        print("New best model saved.")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_losses, label="Train loss")
plt.plot(test_losses, label="Test loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning curves - Loss")
plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_accuracies, label="Train accuracy")
plt.plot(test_accuracies, label="Test accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Learning curves - Accuracy")
plt.legend()
plt.grid()

plt.show()

In [ ]:
loaded_checkpoint = torch.load(model_path, map_location=device)

best_model = ResNet18().to(device)
best_model.load_state_dict(loaded_checkpoint["net"])
best_model.eval()

print("Best model loaded.")
print("Best test accuracy:", loaded_checkpoint["best_test_acc"])

In [ ]:
final_test_loss, final_test_acc, y_true, y_pred = evaluate(
    best_model,
    testloader,
    criterion,
    device
)

print("Final test loss:", final_test_loss)
print("Final test accuracy:", final_test_acc)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 8))
plt.imshow(cm)
plt.title("Confusion matrix")
plt.colorbar()

plt.xticks(range(10), c10train.classes, rotation=45)
plt.yticks(range(10), c10train.classes)

plt.xlabel("Predicted label")
plt.ylabel("True label")

plt.show()

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=c10train.classes
))